# notebook 06 - task 3: genai-augmented reflection

**assignment requirement:** reflect on how ai tools were used for part 1 and part 2.

steps:
1. list all major decisions about the final approach for part 1 and part 2 where ai was used
2. for each decision, describe how you evaluated the ai's suggestion before accepting/rejecting
3. choose the one decision where the most meaningful iterative refinement occurred

In [1]:
import pandas as pd

import sys
sys.path.append("..")
from src.config.settings import DATA_DIR

pd.set_option("display.max_colwidth", 100)

## 1. major decisions where ai was used

below is a table of all major decisions in the final approach, the ai tool used,
how the suggestion was evaluated, and a representative prompt.

In [2]:
decisions = [
    {
        "decision": "filter to agentic-only prs for clustering",
        "part": "part 1",
        "ai_prompt_purpose": "determine whether to include human prs in clustering",
        "ai_suggestion": "cluster only agentic prs to avoid fillna(0) bias from a1 bonus",
        "evaluation": "checked that human prs have zero review data (n_formal_reviews=0, n_review_comments=0) and nan churn. verified that including them would create a trivial human-vs-agentic split rather than meaningful effort groups. ran correlation check confirming the bias.",
        "accepted": "yes",
        "prompt": "human prs have no review data. should i include them in clustering or filter to agentic-only? what are the implications for cluster interpretation?",
    },
    {
        "decision": "define 5 effort dimensions (3 from a1/a2 + 2 new)",
        "part": "part 1",
        "ai_prompt_purpose": "design effort dimensions that are domain-distinct",
        "ai_suggestion": "reuse time_to_merge_hours, n_formal_reviews, n_review_comments from a1/a2. add n_unique_reviewers (breadth) and churn_per_review_cycle (burden) as new dimensions capturing reviewer diversity and cognitive load per review round.",
        "evaluation": "computed spearman correlations. initial suggestion of review_intensity (comments/hour) had rho=0.996 with n_review_comments - essentially the same dimension. rejected and replaced with n_unique_reviewers which had rho=0.863 with n_formal_reviews (moderate, capturing distinct aspect). verified no pair exceeds rho=0.9.",
        "accepted": "partially - rejected review_intensity, accepted n_unique_reviewers instead",
        "prompt": "what are unique dimensions of review effort i can derive from the dataset? i already have time_to_merge, n_formal_reviews, n_review_comments. i need dimensions that capture distinct aspects.",
    },
    {
        "decision": "use kmeans over gmm and dbscan",
        "part": "part 1",
        "ai_prompt_purpose": "choose clustering algorithm given zero-inflated data",
        "ai_suggestion": "kmeans with log1p + standardscaler preprocessing. dbscan will treat the zero-spike as one cluster and label sparse prs as noise. gmm assumes gaussian distributions violated by zero-inflation.",
        "evaluation": "ran kmeans and gmm sweeps for k=2..8. kmeans outperformed gmm on silhouette, dbi, and calinski-harabasz for all k>=3. gmm's poor performance confirmed the gaussian assumption violation. accepted kmeans.",
        "accepted": "yes",
        "prompt": "my clustering data has 77-89% zeros on 3 of 5 dimensions. which clustering algorithm handles zero-inflated data best? kmeans, gmm, or dbscan?",
    },
    {
        "decision": "select k=4 over k=2 (highest silhouette)",
        "part": "part 1",
        "ai_prompt_purpose": "determine optimal k for meaningful cluster profiles",
        "ai_suggestion": "k=2 has highest silhouette (0.618) but only separates reviewed vs not-reviewed. k=4 (sil=0.395) separates auto-merged into small vs large, which has domain meaning.",
        "evaluation": "inspected cluster profiles at k=2,3,4. at k=2, cluster profiles were trivial. at k=4, the additional cluster (auto-merged large) revealed that 45% of prs are large unreviewed code additions - a significant domain finding. accepted k=4.",
        "accepted": "yes",
        "prompt": "k=2 has the best silhouette score but seems too simple. how should i balance silhouette score with domain interpretability when choosing k?",
    },
    {
        "decision": "use tf-idf + logistic regression for text classification",
        "part": "part 2",
        "ai_prompt_purpose": "choose text representation and classifier",
        "ai_suggestion": "tf-idf with bigrams + logistic regression with class_weight=balanced. interpretable, fast, and strong baseline for sparse features.",
        "evaluation": "compared logistic regression and linear svc on the same tf-idf features. lr achieved macro-f1=0.597 vs svc=0.575. lr also provides interpretable coefficients showing which terms predict each cluster. accepted lr.",
        "accepted": "yes",
        "prompt": "what is the best approach for classifying pr descriptions into 4 clusters with severe class imbalance (5-45%)? tf-idf vs embeddings? lr vs svm?",
    },
    {
        "decision": "preprocess pr body text (strip markdown, code blocks, urls)",
        "part": "part 2",
        "ai_prompt_purpose": "design text preprocessing pipeline",
        "ai_suggestion": "lowercase, remove code blocks, strip urls, collapse markdown formatting, remove punctuation, collapse whitespace.",
        "evaluation": "inspected sample cleaned bodies for each cluster. verified that auto-generated content (dependabot, renovate bot signatures) was removed. checked that meaningful technical terms were preserved. accepted.",
        "accepted": "yes",
        "prompt": "pr descriptions contain markdown, code blocks, urls, and bot-generated signatures. how should i clean them for tf-idf classification?",
    },
]

decisions_df = pd.DataFrame(decisions)
decisions_df[["decision", "part", "accepted"]]

,decision,part,accepted
0,filter to agentic-only prs for clustering,part 1,yes
1,define 5 effort dimensions (3 from a1/a2 + 2 new),part 1,"partially - rejected review_intensity, accepted n_unique_reviewers instead"
2,use kmeans over gmm and dbscan,part 1,yes
3,select k=4 over k=2 (highest silhouette),part 1,yes
4,use tf-idf + logistic regression for text classification,part 2,yes
5,"preprocess pr body text (strip markdown, code blocks, urls)",part 2,yes


## 2. detailed evaluation of each decision

In [3]:
for i, row in decisions_df.iterrows():
    print(f"decision {i+1}: {row['decision']}")
    print(f"part: {row['part']}")
    print(f"ai suggestion: {row['ai_suggestion']}")
    print(f"evaluation: {row['evaluation']}")
    print(f"accepted: {row['accepted']}")
    print(f"representative prompt: {row['prompt']}")

decision 1: filter to agentic-only prs for clustering
part: part 1
ai suggestion: cluster only agentic prs to avoid fillna(0) bias from a1 bonus
evaluation: checked that human prs have zero review data (n_formal_reviews=0, n_review_comments=0) and nan churn. verified that including them would create a trivial human-vs-agentic split rather than meaningful effort groups. ran correlation check confirming the bias.
accepted: yes
representative prompt: human prs have no review data. should i include them in clustering or filter to agentic-only? what are the implications for cluster interpretation?
decision 2: define 5 effort dimensions (3 from a1/a2 + 2 new)
part: part 1
ai suggestion: reuse time_to_merge_hours, n_formal_reviews, n_review_comments from a1/a2. add n_unique_reviewers (breadth) and churn_per_review_cycle (burden) as new dimensions capturing reviewer diversity and cognitive load per review round.
evaluation: computed spearman correlations. initial suggestion of review_intensity

## 3. most meaningful iterative refinement

**decision: define effort dimensions (decision 2)**

this is the decision where the most meaningful iterative refinement occurred.

### before (initial ai suggestion)

the initial suggestion included `review_intensity` (n_review_comments /
time_to_merge_hours) as the 4th dimension, capturing 'concentration' of
review effort. this seemed reasonable on domain grounds.


### what was wrong

after computing the spearman correlation matrix, `review_intensity` and
`n_review_comments` had rho = 0.996. they were nearly identical because
most prs have very short durations (median 0.04h), making comments/hour
essentially proportional to raw comment count. including both would violate
the assignment requirement that each dimension capture a 'unique aspect
not already captured by the others.'

### how the revision fixed it

i rejected `review_intensity` and replaced it with `n_unique_reviewers`
(distinct human reviewers per pr). this dimension captures 'breadth' of
reviewer involvement - whether scrutiny comes from many independent
perspectives or a single reviewer. the spearman correlation between
`n_unique_reviewers` and `n_formal_reviews` was 0.863 (moderate, not
extreme), and no pair in the revised set exceeded rho = 0.9. this
refinement is analytically meaningful because it replaces a redundant
dimension with one that captures a genuinely distinct effort aspect:
reviewer diversity vs procedural iterations.

### validation

the correlation check was the key validation: instead of just accepting
the ai's domain reasoning, i ran a quantitative test that revealed the
redundancy. the replacement dimension was then validated by checking
that it had moderate (not extreme) correlation with existing dimensions
and a clear domain interpretation.

export decisions table for the replication package

In [4]:
export_cols = [
    "decision",
    "part",
    "ai_prompt_purpose",
    "prompt",
    "ai_suggestion",
    "evaluation",
    "accepted",
]
decisions_df[export_cols].to_csv(DATA_DIR / "ai_usage_log.csv", index=False)
print(f"saved ai usage log to {DATA_DIR / 'ai_usage_log.csv'}")

saved ai usage log to C:\Users\DPQUAI250113\Downloads\ml\cisc839-assignment-3\data\ai_usage_log.csv
